<img src="MUML_header.svg" alt="Header" style="width:100%; height:auto;">

# 13_01 Recommender systems

In this notebook, you will implement a content based recommender system and a collaborative filtering recommender system using PyTorch.

How this notebook works: every exercise asks you to implement one or more **functions**. Some are partially written (read them and fill the `TODO`s), others only have a signature and a docstring. Below each one there is a check cell with `assert`s: run it to validate your implementation before continuing, because later exercises depend on the earlier ones.


In [1]:
# !pip install torch pandas numpy scikit-learn requests

import copy
import math
import random
import zipfile
import requests
import numpy as np
import pandas as pd

from pathlib import Path
from typing import Callable, Dict, List, Tuple

from sklearn.preprocessing import normalize

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, RandomSampler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

NUM_GENRES = 19
K = 10
BATCH = 1024

## Exercise 1 — Dataset: MovieLens 100K, Download & Files

In this lab, we will use the MovieLens 100K dataset, which includes both explicit feedback (around 100,000 movie ratings) and basic content (genres). Here is the main information about this dataset:
- Dataset: MovieLens 100K  
- Source: GroupLens Research  
- Direct download: [http://files.grouplens.org/datasets/movielens/ml-100k.zip](http://files.grouplens.org/datasets/movielens/ml-100k.zip)  

Contents we need:
- `u.data` — tab-separated file with user, item, rating, timestamp.
- `u.item` — pipe-separated file with movie metadata, including a 19-dim genre one-hot vector we’ll use as content features.


**What to do**
1. Download and extract the ZIP.
2. Load `u.data` and `u.item` into DataFrames.
3. Remap raw IDs to **0-based contiguous** `uid` and `iid` (PyTorch Embedding friendly).
4. Build the item content matrix $X_{\text{item}} \in \mathbb{R}^{\text{num\_items} \times 19}$ from the genre flags.
5. Normalize item rows (L2) so a dot product becomes a cosine similarity later.

> **Tip:** Keep an eye on encodings (`latin-1` for `u.item`) and separators (`\t` vs `|`).  

**Functions to implement**
- `download_movielens(data_dir, url)`
- `load_movielens(data_dir) -> (ratings, items)`
- `remap_ids(ratings, items) -> (ratings, items)`: adds `uid` / `iid` columns to both DataFrames.
- `build_item_matrix(items, num_items) -> np.ndarray`

> **Check:** the cell right below the code runs `assert`s on your functions. They should all pass before you move on.

In [2]:
URL = 'http://files.grouplens.org/datasets/movielens/ml-100k.zip'
DATA_DIR = Path('data/ml-100k')


def download_movielens(data_dir: Path = DATA_DIR, url: str = URL) -> None:
    """Download and extract MovieLens 100K so that `data_dir` contains u.data and u.item."""
    data_dir.mkdir(parents=True, exist_ok=True)
    if (data_dir / 'u.data').exists():
        return
    zip_path = data_dir / 'ml-100k.zip'
    if not zip_path.exists():
        print("Downloading MovieLens 100K...")
        r = requests.get(url, verify=False)
        r.raise_for_status()
        # TODO: save the downloaded content to zip_path
        with open(zip_path, 'wb') as f:
            f.write(r.content)
        
        
    print("Extracting...")
    with zipfile.ZipFile(zip_path) as z:
        # TODO: put every file of the archive directly inside data_dir (the zip contains an 'ml-100k/' folder)
        for member in z.namelist():
            filename = Path(member).name
            if filename:
                source = z.open(member)
                target = open(data_dir / filename, "wb")
                with source, target:
                    target.write(source.read())


def load_movielens(data_dir: Path = DATA_DIR) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Return raw `ratings` (user, item, rating, ts) and `items` (item, title, genre_0 ... genre_18)."""
    ratings = pd.read_csv(data_dir / 'u.data', sep='\t', names=['user', 'item', 'rating', 'ts'])
    # u.item columns (README): movie id | title | release date | video release date | IMDb URL | 19 genre flags
    # TODO: read u.item, keep id, title and the genre flags, and name the columns as documented above
    genre_cols = [f'genre_{i}' for i in range(NUM_GENRES)]
    item_cols = ['item', 'title', 'release_date', 'video_release_date', 'imdb_url'] + genre_cols
    items = pd.read_csv(data_dir / 'u.item', sep='|', names=item_cols, encoding='latin-1')
    items = items[['item', 'title'] + genre_cols]
    return ratings, items

def remap_ids(ratings: pd.DataFrame, items: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Add 0-based contiguous `uid` and `iid` columns. `items` is returned sorted by `iid`."""
    ratings, items = ratings.copy(), items.copy()
    user_mapping = {u: i for i, u in enumerate(sorted(ratings['user'].unique()))}
    ratings['uid'] = ratings['user'].map(user_mapping)

    # TODO: same for items; `ratings` and `items` must share the same iid
    item_mapping = {item: i for i, item in enumerate(sorted(items['item'].unique()))}
    ratings['iid'] = ratings['item'].map(item_mapping)
    items['iid'] = items['item'].map(item_mapping)

    items = items.dropna(subset=['iid'])
    items['iid'] = items['iid'].astype(int)
    return ratings, items.sort_values('iid').reset_index(drop=True)


def build_item_matrix(items: pd.DataFrame, num_items: int) -> np.ndarray:
    """Return X_item, float32 of shape (num_items, NUM_GENRES), row i = genre flags of item iid=i, L2-normalised."""
    genre_cols = [f'genre_{i}' for i in range(NUM_GENRES)]
    X_item = items.sort_values('iid')[genre_cols].values.astype(np.float32)
    X_item = normalize(X_item, norm='l2', axis=1)
    return X_item

In [3]:
download_movielens(DATA_DIR)
ratings, items = load_movielens(DATA_DIR)
ratings, items = remap_ids(ratings, items)
num_users = ratings['uid'].nunique()
num_items = ratings['iid'].nunique()
X_item = build_item_matrix(items, num_items)
print(f"Users: {num_users}, Items: {num_items}, Genres: {NUM_GENRES}")

assert len(ratings) == 100_000
assert ratings['uid'].min() == 0 and ratings['uid'].max() == num_users - 1 == 942
assert ratings['iid'].min() == 0 and ratings['iid'].max() == num_items - 1 == 1681
assert items['iid'].tolist() == list(range(num_items)), "items must be sorted by iid, one row per item"
assert X_item.shape == (num_items, NUM_GENRES) and X_item.dtype == np.float32
norms = np.linalg.norm(X_item, axis=1)
assert np.all((norms < 1e-6) | (np.abs(norms - 1) < 1e-5)), "rows must be unit-norm (or all-zero)"
assert (norms > 0.5).mean() > 0.99
assert items.loc[0, 'title'].startswith('Toy Story')
assert set(np.nonzero(X_item[0])[0]) == {3, 4, 5}, "row 0 must be Toy Story: Animation, Children's, Comedy"
print("Exercise 1: all checks passed")

Users: 943, Items: 1682, Genres: 19
Exercise 1: all checks passed


## Exercise 2 — Train/Validation/Test Partition (Leave-Last-N per User)

Recommenders should predict future interactions from past behavior. Since random splits can leak future information into training, we’ll use a **leave-last-N** strategy *per user*:

- Sort each user’s interactions by timestamp.
- Assign their last 1 interaction to test.
- Assign their second-to-last 1 to validation.
- Put all earlier interactions into train.

This mimics a realistic scenario: we train on what the user has already done and evaluate on what they do next.

Edge cases:
- If a user has only **1** interaction → goes to **test**; user contributes no train/val data.
- If a user has **2** interactions → last to **test**, previous to **val** (nothing left for train). In this notebook we use **val=1, test=1** and put the rest (if any) into train.

**Note:**
- Some users may end up with **no train interactions**; that reflects cold-start users in practice. We’ll typically filter them from evaluation or handle gracefully.

**Functions to implement**
- `leave_last_n_split(df, n_valid=1, n_test=1) -> (train, valid, test)`

> **Check:** the cell right below the code runs `assert`s on your functions. They should all pass before you move on.

In [4]:
def leave_last_n_split(df: pd.DataFrame, n_valid: int = 1, n_test: int = 1):
    """Per-user temporal split of `df` (columns uid, iid, rating, ts).

    For each user, the last `n_test` interactions (by `ts`) go to test, the `n_valid` ones before
    those go to valid and all earlier ones to train. Users with too few interactions fill test
    first, then valid. Returns three DataFrames (copies) that keep the original index.
    """
    df_sorted = df.sort_values(['uid', 'ts'])
    
    test_indices = []
    valid_indices = []
    train_indices = []
    
    for uid, group in df_sorted.groupby('uid'):
        indices = group.index.tolist()
        num_interactions = len(indices)
        
        n_te = min(n_test, num_interactions)
        rem = num_interactions - n_te
        n_va = min(n_valid, rem)
        
        if n_te > 0:
            test_indices.extend(indices[-n_te:])
        if n_va > 0:
            valid_indices.extend(indices[-(n_te + n_va):-n_te])
        if num_interactions > n_te + n_va:
            train_indices.extend(indices[:-(n_te + n_va)])
            
    train = df.loc[train_indices].copy()
    valid = df.loc[valid_indices].copy()
    test = df.loc[test_indices].copy()
    
    return train, valid, test



In [5]:
def items_of(d, u):
    return d[d['uid'] == u].sort_values('ts')['iid'].tolist()

toy = pd.DataFrame({
    'uid':    [0,  0,  0,  0,  0,  1,  2,  2,  3,  3,  3],
    'iid':    [0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10],
    'rating': [3,  4,  5,  1,  2,  5,  4,  4,  3,  2,  1],
    'ts':     [50, 10, 40, 20, 30, 7,  9,  8,  3,  2,  1],
}, index=[100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110])
tr, va, te = leave_last_n_split(toy)
assert items_of(te, 0) == [0] and items_of(va, 0) == [2] and items_of(tr, 0) == [1, 3, 4], "ts order, not row order"
assert items_of(te, 1) == [5] and items_of(va, 1) == [] and items_of(tr, 1) == []
assert items_of(te, 2) == [6] and items_of(va, 2) == [7] and items_of(tr, 2) == []
assert items_of(te, 3) == [8] and items_of(va, 3) == [9] and items_of(tr, 3) == [10]
assert sorted(tr.index.tolist() + va.index.tolist() + te.index.tolist()) == toy.index.tolist()
assert set(tr.columns) == set(toy.columns)

tr2, va2, te2 = leave_last_n_split(toy, n_valid=2, n_test=2)
assert items_of(te2, 0) == [2, 0] and items_of(va2, 0) == [3, 4] and items_of(tr2, 0) == [1]

train, valid, test = leave_last_n_split(ratings, n_valid=1, n_test=1)
for name, d in [('train', train), ('valid', valid), ('test', test)]:
    print(name, d.shape, 'unique users', d['uid'].nunique(), 'unique items', d['iid'].nunique())
assert len(train) + len(valid) + len(test) == len(ratings)
assert len(set(train.index) | set(valid.index) | set(test.index)) == len(ratings)
assert test['uid'].nunique() == num_users and test['uid'].is_unique
assert valid['uid'].is_unique
tr_last = train.groupby('uid')['ts'].max()
va_ts = valid.set_index('uid')['ts']
te_ts = test.set_index('uid')['ts']
assert (va_ts.loc[tr_last.index] >= tr_last).all(), "train must come before valid"
assert (te_ts.loc[va_ts.index] >= va_ts).all(), "valid must come before test"
print("Exercise 2: all checks passed")

train (98114, 6) unique users 943 unique items 1678
valid (943, 6) unique users 943 unique items 534
test (943, 6) unique users 943 unique items 529
Exercise 2: all checks passed


## Exercise 3 — Content-Based Recommender

Since each item has a content vector $ \mathbf{x}_i $ (here: 19-dim genres). We are going to do the following:

- Build a user profile $ \mathbf{u} $ as an aggregation of the content vectors of items the user **liked** (ratings ≥ 4) in train.
- Rank candidate items by cosine similarity $ \cos(\mathbf{u}, \mathbf{x}_i) $ (exclude items seen in train).

**Two user-profile variants**
1. Binary weighting: average of liked item vectors.
2. Deviation weighting: weight each liked item by $ (r_{ui} - \bar r_u)^+ $ to emphasize above-average likes.

**What to do**
- Construct user profiles for all users using train data.
- For each user, score all items and recommend Top-K (filter out items seen in train).

**Functions to implement**
- `seen_items(df) -> {uid: set(iid)}` (`liked_items` is given, as a model)
- `build_user_profiles(X_item, train, num_users, weighting)` -> array `(num_users, n_genres)`
- `recommend_content(U, X_item, user, seen, topk)` -> list of item ids, best first

> **Check:** the cell right below the code runs `assert`s on your functions. They should all pass before you move on.

In [6]:
def liked_items(df: pd.DataFrame, threshold: int = 4) -> Dict[int, set]:
    """{uid: set of iids the user rated >= threshold in df}."""
    return df[df['rating'] >= threshold].groupby('uid')['iid'].apply(set).to_dict()


def seen_items(df: pd.DataFrame) -> Dict[int, set]:
    """{uid: set of all iids the user interacted with in df}."""
    return df.groupby('uid')['iid'].apply(set).to_dict()


def build_user_profiles(X_item: np.ndarray, train: pd.DataFrame, num_users: int,
                        weighting: str = 'binary', threshold: int = 4) -> np.ndarray:
    """User profiles = weighted average of the vectors of the items the user liked in `train`
    (rating >= threshold), L2-normalised so that a dot product is a cosine similarity.

    weighting='binary':        every liked item has weight 1.
    weighting='by_rating_dev': weight (r_ui - mean_u)^+, where mean_u is the mean of ALL the user's
                               train ratings. If every weight of a user is 0, fall back to weight 1.
    Users without liked items get a zero vector. Any other `weighting` raises ValueError.
    Returns a float32 array of shape (num_users, X_item.shape[1]).
    """
    n_genres = X_item.shape[1]
    profiles = np.zeros((num_users, n_genres), dtype=np.float32)
    
    liked_df = train[train['rating'] >= threshold]

    if weighting == 'by_rating_dev':
        user_means = train.groupby('uid')['rating'].mean()
    
    for uid, group in liked_df.groupby('uid'):
        if uid >= num_users:
            continue
        iids = group['iid'].values
        ratings = group['rating'].values
        item_vecs = X_item[iids]
        
        if weighting == 'binary':
            weights = np.ones(len(ratings), dtype=np.float32)
        elif weighting == 'by_rating_dev':
            user_mean = user_means.loc[uid]
            weights = np.maximum(0.0, ratings - user_mean)
            if np.all(weights == 0):
                weights = np.ones(len(ratings), dtype=np.float32)
        else:
            raise ValueError(f"Unknown weighting: {weighting}")
            
        profiles[uid] = np.dot(weights, item_vecs)

    return normalize(profiles, norm='l2', axis=1)


def recommend_content(U: np.ndarray, X_item: np.ndarray, user: int, seen: set = frozenset(),
                      topk: int = 50) -> List[int]:
    """Top-`topk` item ids for `user`, ranked by cosine similarity between profile and items,
    skipping the ids in `seen`. Returns [] if the user's profile is all zeros."""
    profile = U[user]
    
    if np.all(profile == 0):
        return []
        
    scores = np.dot(X_item, profile)

    best_items = np.argsort(scores)[::-1]
    recommendations = [int(i) for i in best_items if i not in seen]
    return recommendations[:topk]

In [7]:
toy_train = pd.DataFrame({'uid': [0, 0, 0, 1, 1, 2], 'iid': [0, 1, 2, 0, 2, 1], 'rating': [5, 4, 1, 2, 3, 5]})
assert seen_items(toy_train) == {0: {0, 1, 2}, 1: {0, 2}, 2: {1}}
assert liked_items(toy_train) == {0: {0, 1}, 2: {1}}

X_toy = np.eye(3, dtype=np.float32)
U_b = build_user_profiles(X_toy, toy_train, 4, 'binary')
assert U_b.shape == (4, 3)
assert np.allclose(U_b[0], [1 / np.sqrt(2), 1 / np.sqrt(2), 0], atol=1e-4)
assert np.allclose(U_b[1], 0) and np.allclose(U_b[3], 0), "no liked items -> zero profile"
assert np.allclose(U_b[2], [0, 1, 0], atol=1e-4)
U_d = build_user_profiles(X_toy, toy_train, 4, 'by_rating_dev')
w = np.array([5 - 10 / 3, 4 - 10 / 3, 0])
assert np.allclose(U_d[0], w / np.linalg.norm(w), atol=1e-4), "mean_u uses all train ratings of the user"
assert np.allclose(U_d[2], [0, 1, 0], atol=1e-4), "all-zero weights -> fall back to uniform"
try:
    build_user_profiles(X_toy, toy_train, 4, 'nope')
    raise AssertionError("an unknown weighting must raise ValueError")
except ValueError:
    pass

X_rec = np.array([[1, 0], [0.6, 0.8], [0, 1], [0.8, 0.6]], dtype=np.float32)
U_rec = np.array([[1, 0], [0, 0]], dtype=np.float32)
assert [int(i) for i in recommend_content(U_rec, X_rec, 0)] == [0, 3, 1, 2]
assert [int(i) for i in recommend_content(U_rec, X_rec, 0, seen={0})] == [3, 1, 2]
assert [int(i) for i in recommend_content(U_rec, X_rec, 0, seen={0}, topk=2)] == [3, 1]
assert list(recommend_content(U_rec, X_rec, 1)) == []

seen_train = seen_items(train)
liked_test = liked_items(test)
U_bin = build_user_profiles(X_item, train, num_users, 'binary')
U_dev = build_user_profiles(X_item, train, num_users, 'by_rating_dev')
for U in (U_bin, U_dev):
    assert U.shape == (num_users, NUM_GENRES)
    n = np.linalg.norm(U, axis=1)
    assert np.all((n < 1e-6) | (np.abs(n - 1) < 1e-4))
    assert (n > 0.5).mean() > 0.9
rec = recommend_content(U_bin, X_item, 0, seen_train[0], topk=50)
assert len(rec) == 50 and not set(rec) & seen_train[0]
print("Exercise 3: all checks passed")

Exercise 3: all checks passed


## Exercise 4 — Ranking Metrics (Precision@K, Recall@K, NDCG@K)

In production recommenders, we care about the *top of the list* quality, not just rating prediction error. We will:

- Mark items with rating ≥ 4 in test as *relevant* for each user.
- Produce Top-K recommendations per user.
- Compute:
  - **Precision@K**: fraction of recommended items in Top-K that are relevant.
  - **Recall@K**: fraction of relevant items we managed to include in Top-K.
  - **NDCG@K**: position-sensitive gain (rewards placing relevant items higher).

- Evaluate the content-based recommender in the previous exercise with P@10, R@10, NDCG@10 using test likes.

**Functions to implement**
- `precision_at_k`, `recall_at_k`, `ndcg_at_k` (binary relevance; in NDCG the ideal ranking puts all relevant items first)
- `evaluate_ranking(recommend_fn, relevant_by_user, num_users, k)` -> mean `(P@k, R@k, NDCG@k)` over users. `recommend_fn(user)` returns a ranked list of item ids. Skip users for which both the recommendation list and the relevant set are empty.

Then evaluate both content-based profiles (binary and rating-deviation) in the cell at the end.

> **Check:** the cell right below the code runs `assert`s on your functions. They should all pass before you move on.

In [8]:
def precision_at_k(recommended: List[int], relevant: set, k: int = 10) -> float:
    """Fraction of the first k recommended items that are relevant."""
    if not relevant or k == 0:
        return 0.0
        
    rec_k = recommended[:k]
    if not rec_k:
        return 0.0
        
    hits = sum(1 for item in rec_k if item in relevant)
    return hits / len(rec_k)


def recall_at_k(recommended: List[int], relevant: set, k: int = 10) -> float:
    """Fraction of the relevant items found in the first k recommended items."""
    if not relevant:
        return 0.0
    rec_k = recommended[:k]
    hits = sum(1 for item in rec_k if item in relevant)
    return hits / len(relevant)


def ndcg_at_k(recommended: List[int], relevant: set, k: int = 10) -> float:
    """NDCG@k with binary relevance and log2(rank + 1) discount; 0.0 if there are no relevant items."""
    if not relevant or k == 0:
        return 0.0
    rec_k = recommended[:k]
    dcg = 0.0
    for idx, item in enumerate(rec_k):
        if item in relevant:
            dcg += 1.0 / math.log2(idx + 2)
            
    ideal_hits = min(k, len(relevant))
    idcg = sum(1.0 / math.log2(idx + 2) for idx in range(ideal_hits))
    
    if idcg == 0.0:
        return 0.0
    return dcg / idcg


def evaluate_ranking(recommend_fn: Callable[[int], List[int]], relevant_by_user: Dict[int, set],
                     num_users: int, k: int = 10) -> Tuple[float, float, float]:
    """Mean (precision, recall, ndcg)@k over users 0..num_users-1."""
    precs, recs, ndcgs = [], [], []
    for u in range(num_users):
        rec = recommend_fn(u)
        rel = relevant_by_user.get(u, set())
        # TODO: skip users with no recommendations and no relevant items; otherwise store their three metrics
        if not rec and not rel:
                continue
        precs.append(precision_at_k(rec, rel, k))
        recs.append(recall_at_k(rec, rel, k))
        ndcgs.append(ndcg_at_k(rec, rel, k))
        
    return float(np.mean(precs)) if precs else 0.0, \
           float(np.mean(recs)) if recs else 0.0, \
           float(np.mean(ndcgs)) if ndcgs else 0.0

In [9]:
approx = lambda a, b: abs(a - b) < 1e-6

rec, rel = [1, 2, 3, 4], {1, 3}
assert approx(precision_at_k(rec, rel, 4), 0.5) and approx(precision_at_k(rec, rel, 2), 0.5) and approx(precision_at_k(rec, rel, 1), 1.0)
assert approx(recall_at_k(rec, rel, 4), 1.0) and approx(recall_at_k(rec, rel, 2), 0.5) and approx(recall_at_k(rec, rel, 1), 0.5)
assert approx(ndcg_at_k(rec, rel, 4), (1 + 1 / math.log2(4)) / (1 + 1 / math.log2(3)))
assert approx(ndcg_at_k([1, 3, 2, 4], rel, 4), 1.0), "a perfect ranking has NDCG 1"
assert approx(ndcg_at_k([5, 1], {1}, 2), 1 / math.log2(3))
assert approx(ndcg_at_k([5, 1], {1}, 1), 0.0) and approx(precision_at_k([5, 1], {1}, 1), 0.0)
assert approx(ndcg_at_k(list(range(20)), set(range(20)), 5), 1.0), "ideal DCG is limited to k hits"
assert recall_at_k(rec, set()) == 0.0 and ndcg_at_k(rec, set()) == 0.0

fake_recs = {0: [1, 2, 3], 1: [1, 2, 3], 2: []}
p, r, n = evaluate_ranking(lambda u: fake_recs[u], {0: {1}, 1: {9}}, 3, k=2)
assert approx(p, 0.25) and approx(r, 0.5) and approx(n, 0.5), "user 2 (nothing recommended, nothing relevant) must be skipped"
print("Exercise 4: all checks passed")

Exercise 4: all checks passed


In [10]:
# Evaluate both content-based variants (U_bin, U_dev) with P@10, R@10 and NDCG@10 on the test likes

# Evaluar U_bin
p_bin, r_bin, n_bin = evaluate_ranking(
    lambda u: recommend_content(U_bin, X_item, u, seen_train.get(u, set()), topk=K),
    liked_test, num_users, k=K
)
print(f"[Content Binary] P@{K}={p_bin:.4f} R@{K}={r_bin:.4f} NDCG@{K}={n_bin:.4f}")

# Evaluar U_dev
p_dev, r_dev, n_dev = evaluate_ranking(
    lambda u: recommend_content(U_dev, X_item, u, seen_train.get(u, set()), topk=K),
    liked_test, num_users, k=K
)
print(f"[Content Deviation] P@{K}={p_dev:.4f} R@{K}={r_dev:.4f} NDCG@{K}={n_dev:.4f}")

[Content Binary] P@10=0.0006 R@10=0.0064 NDCG@10=0.0046
[Content Deviation] P@10=0.0007 R@10=0.0074 NDCG@10=0.0053


## Exercise 5 — PyTorch Dataset for explicit ratings

To prepare data to train a collaborative filtering model, we need a `Dataset` to train models to predict ratings using MSE loss on observed interactions.

**What to do**
- A `Dataset` that yields `(uid, iid, rating)` triples from a DataFrame.
- DataLoaders for train, validation, and test.

**Note**: Although we train with MSE (a *pointwise* objective), we will *evaluate* with ranking metrics to reflect recommender use-cases.

**Functions / classes to implement**
- `RatingsDataset(df)`
- `make_loaders(train, valid, test, batch_size) -> (train_loader, valid_loader, test_loader)`: only the training loader is shuffled.

> **Check:** the cell right below the code runs `assert`s on your functions. They should all pass before you move on.

In [11]:
class RatingsDataset(Dataset):
    """Yields (uid, iid, rating) as (long, long, float32) tensors, one row of `df` per item."""
    def __init__(self, df: pd.DataFrame):
        self.u = torch.tensor(df['uid'].values, dtype=torch.long)
        self.i = torch.tensor(df['iid'].values, dtype=torch.long)
        # TODO: ratings as a float32 tensor
        self.r = torch.tensor(df['rating'].values, dtype=torch.float32)

    def __len__(self):
        return len(self.u)

    def __getitem__(self, idx):
        return self.u[idx], self.i[idx], self.r[idx]


def make_loaders(train: pd.DataFrame, valid: pd.DataFrame, test: pd.DataFrame, batch_size: int = BATCH):
    train_loader = DataLoader(RatingsDataset(train), batch_size=batch_size, shuffle=True)
    valid_loader = DataLoader(RatingsDataset(valid), batch_size=batch_size, shuffle=False)
    # TODO: test loader
    test_loader = DataLoader(RatingsDataset(test), batch_size=batch_size, shuffle=False)
    return train_loader, valid_loader, test_loader

In [12]:
ds = RatingsDataset(train)
assert len(ds) == len(train)
u, i, r = ds[0]
assert u.dtype == torch.long and i.dtype == torch.long and r.dtype == torch.float32
row = train.iloc[0]
assert (u.item(), i.item(), r.item()) == (row['uid'], row['iid'], row['rating'])

train_loader, valid_loader, test_loader = make_loaders(train, valid, test, BATCH)
ub, ib, rb = next(iter(train_loader))
assert ub.shape == ib.shape == rb.shape == (BATCH,)
assert isinstance(train_loader.sampler, RandomSampler)
assert not isinstance(valid_loader.sampler, RandomSampler) and not isinstance(test_loader.sampler, RandomSampler)
assert sum(len(b[0]) for b in valid_loader) == len(valid) and sum(len(b[0]) for b in test_loader) == len(test)
assert next(iter(test_loader))[0].tolist() == test['uid'].values[:BATCH].tolist()
print("Exercise 5: all checks passed")

Exercise 5: all checks passed


## Exercise 6 — Collaborative Filtering (Dot Product)

We will define the model as follows: 
$$
\hat r_{ui} = b + b_u + b_i + \langle \mathbf{p}_u, \mathbf{q}_i \rangle
$$
- $ \mathbf{p}_u $: user embedding, $ \mathbf{q}_i $: item embedding
- $ b_u $, $ b_i $: user/item biases  
- $ b $: global bias (additional bias term for the whole model), 
- **Loss**: MSE over observed ratings (plus L2 via optimizer’s weight decay)

**What to do**
- Implement the collaborative filtering model with embeddings + biases.
- Train for a few epochs (e.g., 5–10).
- Evaluate with ranking metrics (Top-K per user, excluding seen items).

**Tip**: For ranking evaluation, we’ll score every item for a user and exclude items seen in *train*.


**Functions / classes to implement**
- `MF(n_users, n_items, d)`: `forward(u, i)` takes two `LongTensor` of shape `(B,)` and returns the predicted ratings, shape `(B,)`.
- `train_model(...)`: partially written, complete the optimisation step. It keeps the weights of the epoch with the best validation MSE.
- `recommend_cf(model, user, seen, num_items, topk, device)` -> list of item ids, best first. It must work with any model with the `forward(u, i)` interface above.

Evaluation reuses `evaluate_ranking` from Exercise 4.

> **Check:** the cell right below the code runs `assert`s on your functions. They should all pass before you move on.

In [13]:
class MF(nn.Module):
    """Biased matrix factorisation: r_hat = b + b_u + b_i + <p_u, q_i>.
    Embeddings start small (std 0.01) and biases at zero. forward(u, i) -> tensor of shape (B,)."""
    def __init__(self, n_users: int, n_items: int, d: int = 64):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, d)
        self.item_emb = nn.Embedding(n_items, d)
        self.user_bias = nn.Embedding(n_users, 1)
        self.item_bias = nn.Embedding(n_items, 1)
        self.global_bias = nn.Parameter(torch.zeros(1))
        
        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, u, i):
        p = self.user_emb(u)
        q = self.item_emb(i)
        b_u = self.user_bias(u).squeeze(1)
        b_i = self.item_bias(i).squeeze(1)
        interaction = (p * q).sum(dim=1)
        return self.global_bias + b_u + b_i + interaction


@torch.no_grad()
def evaluate_mse(model: nn.Module, loader: DataLoader, device: str = 'cpu') -> float:
    model.to(device).eval()
    total = 0.0
    for u, i, r in loader:
        u, i, r = u.to(device), i.to(device), r.to(device)
        total += nn.functional.mse_loss(model(u, i), r, reduction='sum').item()
    return total / max(1, len(loader.dataset))


def train_model(model, train_loader, valid_loader, epochs=10, lr=1e-2, wd=1e-6, device='cpu'):
    """Adam + MSE. Returns the model loaded with the weights of the best validation epoch."""
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    loss_fn = nn.MSELoss()
    best_state, best_val = None, float('inf')
    for ep in range(1, epochs + 1):
        model.train()
        total = 0.0
        for u, i, r in train_loader:
            u, i, r = u.to(device), i.to(device), r.to(device)
            # TODO: one optimisation step; accumulate the summed batch loss in `total`
            opt.zero_grad()
            preds = model(u, i)
            loss = loss_fn(preds, r)
            loss.backward()
            opt.step()
            total += loss.item() * len(u)
        val_loss = evaluate_mse(model, valid_loader, device)
        print(f"Epoch {ep:02d} | train MSE ~ {total / len(train_loader.dataset):.4f} | valid MSE ~ {val_loss:.4f}")
        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    if best_state is not None:
        model.load_state_dict(best_state)
    return model


@torch.no_grad()
def recommend_cf(model: nn.Module, user: int, seen: set, num_items: int, topk: int = 100,
                 device: str = 'cpu') -> List[int]:
    """Score all items for `user` with `model`, drop the ids in `seen`, return the best `topk` ids."""
    model.to(device).eval()
    u_tensor = torch.full((num_items,), user, dtype=torch.long, device=device)
    i_tensor = torch.arange(num_items, dtype=torch.long, device=device)
    scores = model(u_tensor, i_tensor).cpu().numpy()
    
    for item in seen:
        if item < num_items:
            scores[item] = -np.inf
            
    best_items = np.argsort(scores)[::-1]
    recommendations = [int(i) for i in best_items if scores[i] > -np.inf]
    return recommendations[:topk]


In [14]:
def check_rating_model(make_model, data=None, tol=0.1, attempts=3):
    """Generic check for any model with forward(u, i): output shape and ability to fit a few ratings.
    `make_model` builds a fresh model; a few seeds are tried because a bad random init can stall training."""
    if data is None:
        data = ([0, 1, 2, 3, 4, 5], [0, 1, 2, 3, 4, 5], [1.0, 2.0, 3.0, 4.0, 5.0, 3.5])
    u, i, r = (torch.tensor(x) for x in data)
    for seed in range(attempts):
        torch.manual_seed(seed)
        model = make_model()
        assert model(u, i).shape == u.shape, f"forward must return shape (B,), got {tuple(model(u, i).shape)}"
        opt = torch.optim.Adam(model.parameters(), lr=0.02)
        for _ in range(500):
            loss = nn.functional.mse_loss(model(u, i), r)
            opt.zero_grad()
            loss.backward()
            opt.step()
        if loss.item() < tol:
            return
    raise AssertionError(f"the model cannot fit these {len(r)} ratings (loss {loss.item():.3f})")


class _ScoreIsItemId(nn.Module):
    def forward(self, u, i):
        return i.float()

class _ScoreNearUser(nn.Module):
    def forward(self, u, i):
        return -(i - u).abs().float() + 0.01 * i.float()

assert recommend_cf(_ScoreIsItemId(), 3, {9, 8}, 10, topk=3) == [7, 6, 5]
assert recommend_cf(_ScoreIsItemId(), 3, set(), 10, topk=20) == list(range(9, -1, -1))
assert recommend_cf(_ScoreNearUser(), 4, set(), 10, topk=3) == [4, 5, 3], "the user id must be fed to the model"

mf_small = MF(10, 20, d=8)
assert sum(p.numel() for p in mf_small.parameters()) == 10 * 8 + 20 * 8 + 10 + 20 + 1
assert torch.allclose(mf_small(torch.tensor([0, 1]), torch.tensor([2, 3])), torch.zeros(2), atol=0.01)
check_rating_model(lambda: MF(10, 20, d=8))
xor = ([0, 0, 1, 1], [0, 1, 0, 1], [5.0, 1.0, 1.0, 5.0])
check_rating_model(lambda: MF(10, 20, d=8), data=xor, tol=0.5)  # biases alone cannot fit this: needs the user-item interaction
print("Exercise 6 (model and helpers): all checks passed")

Exercise 6 (model and helpers): all checks passed


In [15]:
mf = MF(num_users, num_items, d=64)
mse_before = evaluate_mse(mf, valid_loader, device)
mf = train_model(mf, train_loader, valid_loader, epochs=5, lr=5e-3, wd=1e-6, device=device)
assert evaluate_mse(mf, valid_loader, device) < mse_before, "training did not reduce the validation MSE"

p, r, n = evaluate_ranking(lambda u: recommend_cf(mf, u, seen_train.get(u, set()), num_items, topk=100, device=device),
                           liked_test, num_users, k=K)
print(f"[MF Dot] P@{K}={p:.4f} R@{K}={r:.4f} NDCG@{K}={n:.4f}")

Epoch 01 | train MSE ~ 6.5060 | valid MSE ~ 1.4005
Epoch 02 | train MSE ~ 0.9193 | valid MSE ~ 1.0504
Epoch 03 | train MSE ~ 0.8232 | valid MSE ~ 0.9940
Epoch 04 | train MSE ~ 0.7641 | valid MSE ~ 0.9754
Epoch 05 | train MSE ~ 0.7028 | valid MSE ~ 0.9679
[MF Dot] P@10=0.0022 R@10=0.0223 NDCG@10=0.0097


## Exercise 7 —  Neural Collaborative Filtering (NCF)

Dot-product is a bilinear interaction, which limits the flexibility of combining user and item embeddings. We can model nonlinear interactions by feeding the concatenated user/item embeddings into an **MLP**:

$$
\hat r_{ui} = b + \text{MLP}\left([\mathbf{p}_u \, \| \, \mathbf{q}_i]\right)
$$

**What to do**
- Implement `NeuralCF` with `nn.Embedding` for users/items and an MLP (ReLU layers).
- Train with MSE (as before).
- Evaluate with ranking metrics.


**Classes to implement**
- `NeuralCF(n_users, n_items, d, hidden)`: `hidden` is the tuple of hidden layer sizes (ReLU after each).

Then train it (5 epochs, `lr=5e-3`) and evaluate it exactly as the MF model, in the last cell.

> **Check:** the cell right below the code runs `assert`s on your functions. They should all pass before you move on.

In [16]:
class NeuralCF(nn.Module):
    """r_hat = b + MLP([p_u || q_i]), with `hidden` = sizes of the hidden layers (ReLU after each)
    and a final Linear to 1 output. Embeddings start with std 0.01. forward(u, i) -> shape (B,)."""
    def __init__(self, n_users: int, n_items: int, d: int = 64, hidden: Tuple[int, ...] = (128, 64)):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, d)
        self.item_emb = nn.Embedding(n_items, d)
        self.global_bias = nn.Parameter(torch.zeros(1))
        
        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)
        
        layers = []
        input_dim = d * 2
        for h_dim in hidden:
            layers.append(nn.Linear(input_dim, h_dim))
            layers.append(nn.ReLU())
            input_dim = h_dim
        layers.append(nn.Linear(input_dim, 1))
        
        self.mlp = nn.Sequential(*layers)

    def forward(self, u, i):
        p = self.user_emb(u)
        q = self.item_emb(i)
        x = torch.cat([p, q], dim=1)
        out = self.mlp(x).squeeze(1)
        return self.global_bias + out

In [17]:
ncf_small = NeuralCF(10, 20, d=8, hidden=(16, 8))
assert sum(isinstance(m, nn.ReLU) for m in ncf_small.modules()) == 2
assert sum(isinstance(m, nn.Linear) for m in ncf_small.modules()) == 3
check_rating_model(lambda: NeuralCF(10, 20, d=8, hidden=(16, 8)))
check_rating_model(lambda: NeuralCF(10, 20, d=4, hidden=()))
print("Exercise 7: all checks passed")

Exercise 7: all checks passed


In [18]:
# Train NeuralCF (d=64, hidden=(128, 64), 5 epochs, lr=5e-3) and report P@10, R@10, NDCG@10
ncf = NeuralCF(num_users, num_items, d=64, hidden=(128, 64))
mse_before_ncf = evaluate_mse(ncf, valid_loader, device)
ncf = train_model(ncf, train_loader, valid_loader, epochs=5, lr=5e-3, wd=1e-6, device=device)

p, r, n = evaluate_ranking(
    lambda u: recommend_cf(ncf, u, seen_train.get(u, set()), num_items, topk=100, device=device),
    liked_test, num_users, k=K
)
print(f"[NeuralCF] P@{K}={p:.4f} R@{K}={r:.4f} NDCG@{K}={n:.4f}")

Epoch 01 | train MSE ~ 2.1872 | valid MSE ~ 1.0248
Epoch 02 | train MSE ~ 0.8840 | valid MSE ~ 1.0080
Epoch 03 | train MSE ~ 0.8532 | valid MSE ~ 0.9778
Epoch 04 | train MSE ~ 0.8344 | valid MSE ~ 0.9627
Epoch 05 | train MSE ~ 0.8175 | valid MSE ~ 0.9957
[NeuralCF] P@10=0.0010 R@10=0.0095 NDCG@10=0.0029


## Exercise 8 — Hybrid: Content-Augmented Item Representation

The idea of concatenating item and user features enables the dimension of the each kind of embedding to be different. Thus, we can leverage **content features** (genres) *together with* learned item embeddings.

**A possible strategy**
1. Prepare a content matrix.  
   We’ll use `Xp ∈ ℝ^{num_items × d_content}`, here simply `Xp = X_item` (`d_content = 19`, no dimensionality reduction).
2. Define a model that **concatenates**:
   - user embedding $ \mathbf{p}_u $
   - learned item embedding $ \mathbf{q}_i $
   - content-derived vector $ \mathbf{c}_i = \text{content\_W}(Xp[i]) $ (the matrix $\text{content\_W}$ must start as the identity matrix).
3. Two training regimes:
   - **Frozen content**: `content_W` **does not update** (acts like a fixed projection).
   - **Trainable content**: `content_W` **is learnable**, allowing end-to-end adaptation.

**What to do**
- Implement `HybridNCF`. It receives `Xp` in its constructor and keeps it as part of the model (a buffer, so it follows `.to(device)`), which lets `forward(u, i)` keep the same interface as the previous models and reuse `train_model`, `recommend_cf` and `evaluate_ranking` unchanged.
- Train and evaluate both **frozen** and **trainable** content variants.

**Classes to implement**
- `HybridNCF(n_users, n_items, Xp, d_user, d_item, hidden, freeze_content)`: the attribute holding the projection must be called `content_W` (`nn.Linear`, no bias).

> **Check:** the cell right below the code runs `assert`s on your functions. They should all pass before you move on.

In [19]:
class HybridNCF(nn.Module):
    """MLP over [p_u || q_i || c_i] with c_i = content_W(Xp[i]), plus a global bias.
    content_W: Linear(d_content, d_content, bias=False), starts as the identity and is trainable
    only if `freeze_content` is False. `Xp` (n_items, d_content) is stored inside the model.
    forward(u, i) -> shape (B,)."""
    def __init__(self, n_users: int, n_items: int, Xp, d_user: int = 64, d_item: int = 48,
                 hidden: Tuple[int, ...] = (128, 64), freeze_content: bool = True):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, d_user)
        self.item_emb = nn.Embedding(n_items, d_item)
        
        d_content = Xp.shape[1]
        self.content_W = nn.Linear(d_content, d_content, bias=False)
        self.content_W.weight.data.copy_(torch.eye(d_content))
        self.content_W.weight.requires_grad = not freeze_content
        
        if isinstance(Xp, np.ndarray):
            Xp = torch.tensor(Xp, dtype=torch.float32)
        self.register_buffer('Xp', Xp)
        
        self.global_bias = nn.Parameter(torch.zeros(1))
        
        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)
        
        layers = []
        input_dim = d_user + d_item + d_content
        for h_dim in hidden:
            layers.append(nn.Linear(input_dim, h_dim))
            layers.append(nn.ReLU())
            input_dim = h_dim
        layers.append(nn.Linear(input_dim, 1))
        
        self.mlp = nn.Sequential(*layers)

    def forward(self, u, i):
        p = self.user_emb(u)
        q = self.item_emb(i)
        c = self.content_W(self.Xp[i])
        x = torch.cat([p, q, c], dim=1)
        out = self.mlp(x).squeeze(1)
        return self.global_bias + out

In [20]:
g = torch.Generator().manual_seed(0)
Xp_toy = torch.rand(20, 5, generator=g)

frozen = HybridNCF(10, 20, Xp_toy, d_user=4, d_item=3, hidden=(8,), freeze_content=True)
trainable = HybridNCF(10, 20, Xp_toy, d_user=4, d_item=3, hidden=(8,), freeze_content=False)
for m in (frozen, trainable):
    assert torch.equal(m.content_W.weight, torch.eye(5)), "content_W must start as the identity"
assert not frozen.content_W.weight.requires_grad and trainable.content_W.weight.requires_grad
assert 'Xp' in dict(frozen.named_buffers()), "Xp should be a buffer of the model"

for m, should_change in [(frozen, False), (trainable, True)]:
    m = copy.deepcopy(m)
    opt = torch.optim.Adam([p for p in m.parameters() if p.requires_grad], lr=0.05)
    u, i = torch.tensor([0, 1, 2]), torch.tensor([3, 4, 5])
    for _ in range(5):
        loss = nn.functional.mse_loss(m(u, i), torch.tensor([5.0, 1.0, 3.0]))
        opt.zero_grad()
        loss.backward()
        opt.step()
    changed = not torch.equal(m.content_W.weight, torch.eye(5))
    assert changed == should_change, "content_W should " + ("" if should_change else "not ") + "change during training"

blind = copy.deepcopy(frozen)
blind.item_emb.weight.data.zero_()
out = blind(torch.tensor([0, 0]), torch.tensor([1, 2]))
assert not torch.allclose(out[0], out[1]), "items with different content must get different scores"

for freeze in (True, False):
    check_rating_model(lambda: HybridNCF(10, 20, Xp_toy, d_user=4, d_item=3, hidden=(8,), freeze_content=freeze))
print("Exercise 8: all checks passed")

Exercise 8: all checks passed


In [21]:
Xp = X_item

# Train and evaluate the frozen and the trainable variants (HybridNCF(d_user=64, d_item=48, hidden=(128, 64)), 5 epochs, lr=5e-3)
frozen_model = HybridNCF(num_users, num_items, Xp, d_user=64, d_item=48, hidden=(128, 64), freeze_content=True)
frozen_model = train_model(frozen_model, train_loader, valid_loader, epochs=5, lr=5e-3, wd=1e-6, device=device)
p_f, r_f, n_f = evaluate_ranking(
    lambda u: recommend_cf(frozen_model, u, seen_train.get(u, set()), num_items, topk=100, device=device),
    liked_test, num_users, k=K
)
print(f"[HybridNCF Frozen] P@{K}={p_f:.4f} R@{K}={r_f:.4f} NDCG@{K}={n_f:.4f}")

# Train and evaluate trainable variant
trainable_model = HybridNCF(num_users, num_items, Xp, d_user=64, d_item=48, hidden=(128, 64), freeze_content=False)
trainable_model = train_model(trainable_model, train_loader, valid_loader, epochs=5, lr=5e-3, wd=1e-6, device=device)
p_t, r_t, n_t = evaluate_ranking(
    lambda u: recommend_cf(trainable_model, u, seen_train.get(u, set()), num_items, topk=100, device=device),
    liked_test, num_users, k=K
)
print(f"[HybridNCF Trainable] P@{K}={p_t:.4f} R@{K}={r_t:.4f} NDCG@{K}={n_t:.4f}")

Epoch 01 | train MSE ~ 1.9264 | valid MSE ~ 1.0325
Epoch 02 | train MSE ~ 0.8823 | valid MSE ~ 1.0058
Epoch 03 | train MSE ~ 0.8715 | valid MSE ~ 1.0015
Epoch 04 | train MSE ~ 0.8625 | valid MSE ~ 0.9986
Epoch 05 | train MSE ~ 0.8481 | valid MSE ~ 0.9859
[HybridNCF Frozen] P@10=0.0001 R@10=0.0011 NDCG@10=0.0005
Epoch 01 | train MSE ~ 2.0698 | valid MSE ~ 1.0178
Epoch 02 | train MSE ~ 0.8834 | valid MSE ~ 1.0082
Epoch 03 | train MSE ~ 0.8709 | valid MSE ~ 1.0079
Epoch 04 | train MSE ~ 0.8583 | valid MSE ~ 0.9961
Epoch 05 | train MSE ~ 0.8429 | valid MSE ~ 0.9955
[HybridNCF Trainable] P@10=0.0004 R@10=0.0042 NDCG@10=0.0015


## Exercise 9 — Optional extension exercises

Here is a list of things you could try next. 

1. **Content-based variants**
   - Implement recency-weighted profiles: weight each liked item by $ \alpha^{\Delta t} $ where $ \Delta t $ is time since interaction.
   - Replace genre vectors with TF–IDF of titles and compare.

2. **Dot product variants**
   - Sweep embedding dimension $ d \in \{16, 32, 64, 128\} $.
   - Increase weight decay and observe regularization effects.

3. **Neural CF variants**
   - Deeper/wider MLPs, add dropout, compare performance.
   - Switch to a pairwise ranking loss (BPR) with sampled negatives.

4. **Evaluation improvements**
   - Time-aware candidate filtering: only recommend items released before the user’s last interaction.
   - Add diversity and novelty metrics (e.g., intra-list diversity, popularity-penalized metrics).